## The corpus

The corpus holds Onqelos, Targum Pseudo-Jonathan, the Fragment
Targums, Cairo Genizah fragments, and Targum Neofiti. Neofiti is represented in two views, a
'base' text that keeps only unmarked segments, excluding variants and interlinear or in-line
additions, and a 'full' text that places every segment side by side. The text is Jewish Aramaic (Babylonian for Onqelos, Palestinian for everything else), carried in ETCBC transliteration and in Aramaic square script.


In [1]:
# Loads the dataset and reports its provenance. The version string, the version of the XML the
# dataset was compiled from, and the compilation timestamp are read from the metadata of the
# `otype` feature, which every Text-Fabric dataset carries. The node counts printed here are the
# quantities every later test is measured against.
#
# The corpus is loaded through context-fabric rather than through the advanced application layer
# of Text-Fabric, and three consequences follow from that.
#
# First, the corpus is addressed by path rather than by organisation and repository.
# `cfabric.Fabric(locations, modules)` reads the versioned dataset directory directly, so the
# notebook depends neither on an environment resolving a repository name nor on an environment
# file existing at all. That is what allows this file to run unchanged in the public repository.
#
# Second, context-fabric loads from its own binary cache, which retains feature values and
# discards the header block that carries the metadata. The metadata available through the library
# is therefore empty, and the provenance reported below depends on it, so `tf_meta` reads the
# header out of the shipped feature file itself. That is not a workaround. It reads the artefact
# being published rather than a library's parse of it, so it cannot disagree with what ships. The
# binary cache also holds no staleness check against the feature files beside it, which is why
# the pipeline deletes it after every compile, and reading the files directly is immune to the
# question either way.
#
# Third, there is no advanced application layer, so the helper defined below replaces it and
# prints rather than rendering markup, which keeps the output legible in a terminal and in a
# difference listing.
#
# The version is deliberately not pinned in this notebook. The declared version of the dataset is
# read from the application configuration, which is the same value the compiler writes into the
# metadata of every feature, so this notebook cannot describe an older build than the repository
# ships. An earlier version pinned the string as a literal and continued to report itself as
# current across a rebuild, which would have described a superseded corpus throughout.

import os
from pathlib import Path

import yaml
from dotenv import load_dotenv

import cfabric

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config/xml.yml").exists())
load_dotenv(REPO_ROOT / ".env")

TF_VERSION = yaml.safe_load((REPO_ROOT / "app/config.yaml").read_text())["provenanceSpec"]["version"]
TF_DIR = REPO_ROOT / "tf" / TF_VERSION

# Kept for the provenance line only. Loading no longer goes through them.
ORG = os.environ.get("TF_ORG") or "etcbc"
REPO = os.environ.get("TF_REPO") or "targum"

# The silence level is passed to both calls, because the loader takes its own. Without it the
# library logs one line per feature as it loads, which is longer than most of the output below
# and conveys nothing that the feature inventory does not. The accepted levels are "verbose",
# "auto", "terse" and "deep".
CF = cfabric.Fabric(locations=str(REPO_ROOT / "tf"), modules=TF_VERSION, silent="deep")
api = CF.loadAll(silent="deep")
F, E, L, T, S = api.F, api.E, api.L, api.T, api.S
Fs, Es = api.Fs, api.Es
TF_FEATURES = set(CF.features)


def tf_meta(feature="otype"):
    """The `@`-metadata of one feature, read from the shipped `.tf` file.

    cfabric serves feature data out of its `.cfm` cache, which stores values and drops the `@`
    header, so `CF.features[f].metaData` is empty. Every `.tf` file opens with its own metadata
    block, terminated by the first line that does not start with `@`, so reading it is four lines
    and is authoritative about the artefact rather than about a cache of it.
    """
    out = {}
    path = TF_DIR / f"{feature}.tf"
    if not path.exists():
        return out
    with path.open(encoding="utf8") as fh:
        for line in fh:
            if not line.startswith("@"):
                break
            if "=" in line:
                key, value = line[1:].rstrip("\n").split("=", 1)
                out[key] = value
    return out


def plain(node, withNodes=True):
    """Print a node and the words it contains, in place of the app layer's `pretty`.

    One line of heading and one line per word: the vocalised surface, the lexeme, the part of
    speech and `source`, which together are what the failing cells below are inspected for.
    """
    otype = F.otype.v(node)
    head = f"{otype} {node}" if withNodes else otype
    bits = []
    for name in ("id", "reading", "type", "verse", "chapter", "book", "text"):
        if name in TF_FEATURES:
            value = Fs(name).v(node)
            if value is not None:
                bits.append(f"{name}={value!r}")
    print(f"{head}  {'  '.join(bits)}")
    for w in L.d(node, otype="word") if otype != "word" else (node,):
        print(f"    {w:>8}  {F.voc.v(w)!r:<22} lex={F.lex.v(w)!r:<14} "
              f"sp={F.sp.v(w)!r:<8} source={F.source.v(w)!r}")


meta = tf_meta("otype")


In [2]:
# Checks the dataset's provenance against the files it was compiled from. The dataset is stale
# unless both checks below hold.
#
#   1  The xmlVersion metadata against the newest directory under `xml/`. A dataset compiled from
#      an older XML directory than the one currently present is behind by at least that much.
#   2  The feature descriptions against `config/xml.yml`. A mismatch here means the dataset was
#      compiled before the current feature documentation was written.
#
# A failure in either check means the figures below describe an outdated build of the dataset.

import re
import yaml

FRESH = True


def stale(what, got, want):
    global FRESH
    FRESH = False
    print(f"  STALE  {what}\n           dataset: {got}\n           source:  {want}")


xml_versions = sorted(p.name for p in (REPO_ROOT / "xml").iterdir() if p.is_dir())
newest_xml = xml_versions[-1] if xml_versions else None
print(f"xmlVersion {meta.get('xmlVersion')}  written {meta.get('dateWritten')}")
print(f"newest xml/ {newest_xml}  (of {len(xml_versions)})")
if newest_xml and meta.get("xmlVersion") != newest_xml:
    stale("the dataset was compiled from an older XML than xml/ holds",
          meta.get("xmlVersion"), newest_xml)

declared = yaml.safe_load((REPO_ROOT / "config/xml.yml").read_text())["featureDescriptions"]


def norm(s):
    return re.sub(r"\s+", " ", (s or "")).strip()


for name, want in declared.items():
    if name not in TF_FEATURES:
        continue
    # `tf_meta` reads the header of the feature file directly. The metadata exposed by the
    # library is empty, because it serves from a binary cache that retains values and discards
    # the header block, so this
    # check would have reported all 58 descriptions as stale.
    got = tf_meta(name).get("description")
    if norm(got) != norm(want):
        stale(f"feature {name!r} carries a description config/xml.yml no longer declares",
              norm(got)[:110], norm(want)[:110])

counts = {t: n for t, n in ((t, len(list(F.otype.s(t)))) for t in F.otype.all)}
print("nodes:", ", ".join(f"{t} {counts[t]:,}" for t in F.otype.all))

print()
print("dataset is current" if FRESH else "DATASET IS STALE, rebuild before reading on")
assert FRESH, "stale dataset: see the report above"

xmlVersion 2026-10-28  written 2026-10-29T00:51:18+00:00Z
newest xml/ 2026-10-28  (of 1)
nodes: text 47, book 91, chapter 1,225, lexeme 7,023, verse 25,447, segment 70,389, word_group 461,201, word 624,612

dataset is current


In [3]:
# Reports the feature inventory and the distribution of the three features that the tests below
# depend on. `source` determines which words carry a morphological analysis, `reading` identifies
# the readings that compete for one position, and the segment `type` feature records the editorial
# state of a run of the line.

FEATURES = sorted(f for f in TF_FEATURES if not f.startswith("o") and f != "otext")
for otype in F.otype.all:
    sample = F.otype.s(otype)[:400]
    present = sorted(f for f in FEATURES
                     if hasattr(F, f) and any(Fs(f).v(n) is not None for n in sample))
    print(f"{otype:<11} {', '.join(present)}")
print()
print("source     ", dict(F.source.freqList()))
print("reading    ", dict(F.reading.freqList()))
print("segment    ", dict(F.type.freqList()))
print()
for fmt in sorted(T.formats):
    print(f"  {fmt:<16} {T.text(range(1, 9), fmt=fmt)}")

text        manuscript, text, view
book        book, id
chapter     chapter
lexeme      freq_lex, lex, lex0, lex_utf8, rank_lex
verse       line, line_utf8, verse
segment     end_idx, id, rank, segment, segment_utf8, start_idx, type
word_group  g_cons, g_cons_utf8, g_voc, g_voc_utf8, id, rank, reading
word        cons, cons_utf8, definition, description, end_idx, gloss, lex, lex0, lex_utf8, nu, part, root, source, sp, st, start_idx, trailer, uid, voc, voc_utf8, vs

source      {'word': 621024, 'word_group': 2056, 'none': 1532}
reading     {0: 449163, 1: 11667, 2: 367, 3: 4}
segment     {'unmarked_text': 43751, 'variant': 15270, 'addition': 3638, 'marked_text': 3295, 'editorial_deletion': 2275, 'broken_text': 832, 'marginal': 740, 'interlinear': 240, 'scribal_deletion': 213, 'incomplete': 131, 'source': 4}

  text-cons        מן אוולא ברא אלקים ית שׁמייא וית ארעא
  text-orig-full   מן אוולא ברא אלקים ית שׁמייא וית ארעא
  text-trans-cons  MN >WWL> BR> >LQJM JT CMJJ> WJT >R<>
  text-trans

In [ ]:
# Defines the material shared by every test below.
#
# `letters` is the comparison class used throughout. It strips whitespace so that strings drawn
# from different tiers of the corpus, which do not always agree on how much space they record, can
# still be compared fairly.
#
# `verse_nodes` keys each verse by the pair (manuscript, uid). The `uid` feature is unique within
# one corpus and repeats across corpora, so the manuscript name is needed to make the key unique.
# Two verses of the Genizah material occupy more than one verse node, because the order in which
# the dataset publishes their words does not keep the rows of those verses contiguous. The tests
# below aggregate over the list of nodes that share a key.
#
# `table` prints one row per corpus with a total, and takes derived percentage columns as
# (label, numerator, denominator) triples.

import re
from collections import Counter, defaultdict

_WS = re.compile(r"\s+")


def letters(s):
    return _WS.sub("", s or "")


CORPORA = ["cairo_genizah", "frag_targ", "neofiti_base_raw", "neofiti_full", "psj","onq"]
_BY_NAME = {"neofiti_base": "neofiti_base_raw", "neofiti_full": "neofiti_full",
            "pseudo_jonathan": "psj","onqelos":"onq"}


def corpus_of(text_node):
    name = F.text.v(text_node)
    if name.startswith("cairo"):
        return "cairo_genizah"
    if name.startswith("fragment_targum"):
        return "frag_targ"
    return _BY_NAME[name]


VERSE_KEY_COLUMNS = ("text_text", "book_book", "chapter_chapter", "verse_verse")

verse_nodes = defaultdict(list)
corpus_of_verse = {}
for _t in F.otype.s("text"):
    _name, _corpus = F.text.v(_t), corpus_of(_t)
    for _v in L.d(_t, otype="verse"):
        _k = (_name, F.book.v(L.u(_v, otype="book")[0]),
              F.chapter.v(L.u(_v, otype="chapter")[0]), F.verse.v(_v))
        verse_nodes[_k].append(_v)
        corpus_of_verse[_k] = _corpus


def table(title, rows, cols, pct=(), key="corpus"):
    keys = [k for k in CORPORA if k in rows] + [k for k in rows if k not in CORPORA]
    totals = Counter()
    for k in keys:
        totals.update(rows[k])
    head = [key, *cols, *(lbl for lbl, _, _ in pct)]
    body = []
    for k in [*keys, "TOTAL"]:
        r = totals if k == "TOTAL" else rows[k]
        body.append([k, *(f"{r[c]:,}" for c in cols),
                     *(f"{100.0 * r[n] / r[d]:.2f}" if r[d] else "-" for _, n, d in pct)])
    w = [max(len(str(x)) for x in col) for col in zip(head, *body)]
    rule = "  ".join("-" * i for i in w)
    print(title)
    print("  " + "  ".join(h.rjust(i) for h, i in zip(head, w)))
    print("  " + rule)
    for line in body[:-1]:
        print("  " + "  ".join(str(x).rjust(i) for x, i in zip(line, w)))
    print("  " + rule)
    print("  " + "  ".join(str(x).rjust(i) for x, i in zip(body[-1], w)))


results = {}
print(f"{len(verse_nodes):,} verses distributed over "
      f"{sum(len(v) for v in verse_nodes.values()):,} verse nodes")